# Mechanistic Bridge — Hippo/YAP Activity ↔ Dysbiosis Core Signature

## Central hypothesis
If dysbiosis promotes tumorigenic programs, then:

1. Hippo/YAP target activity should be **higher in IBD** than controls
2. Samples with **higher Hippo/YAP activity** should also show
   **higher expression of the shared IBD–TCGA core signature**

This notebook tests BOTH points quantitatively.

## What this notebook produces
- Boxplot: Hippo/YAP score (IBD vs Control)
- Correlation: Hippo/YAP score vs Core Signature score
- Scatter plot + statistics

These are *mechanistic*, not just descriptive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/GeneticCodonShared/Hippo_Dysbiosis"

# inputs
IBD_LOG     = f"{BASE}/data_processed/ibd/ibd_log2_tpm.csv"
IBD_META    = f"{BASE}/data_processed/ibd/ibd_metadata_clean.csv"
CORE_GENES  = f"{BASE}/results/novelty/core_signature_genes.csv"

# outputs
OUT = f"{BASE}/results/mechanism"
FIG = f"{BASE}/results/figures/mechanism"

import os
os.makedirs(OUT, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

print("Paths loaded")


**Imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr


**Load Data**

In [ ]:
expr = pd.read_csv(IBD_LOG, index_col=0)     # genes x samples
meta = pd.read_csv(IBD_META)
core = pd.read_csv(CORE_GENES)["core_signature_genes"].astype(str).tolist()

print("Expression:", expr.shape)
print("Metadata:", meta.shape)
print("Core genes:", len(core))

meta = meta.set_index("sample_id").loc[expr.columns].reset_index()
display(meta["group"].value_counts())


**Define Hippo/YAP target genes**

In [ ]:
YAP_TARGETS = [
    "CTGF","CYR61","ANKRD1","AREG",
    "AMOTL2","BIRC5","FSTL1"
]

# keep only genes present
YAP_TARGETS = [g for g in YAP_TARGETS if g in expr.index]
print("YAP targets used:", YAP_TARGETS)


**Scoring functions (simple + robust)**

In [ ]:
def zscore_rows(expr_df):
    mu = expr_df.mean(axis=1)
    sd = expr_df.std(axis=1).replace(0, np.nan)
    return expr_df.sub(mu, axis=0).div(sd, axis=0)

def geneset_score(expr_log_df, genes):
    genes = [g for g in genes if g in expr_log_df.index]
    Z = zscore_rows(expr_log_df.loc[genes]).fillna(0)
    return Z.mean(axis=0)


# **Compute Hippo/YAP score per IBD sample**

In [ ]:
hippo_score = geneset_score(expr, YAP_TARGETS)

scores = pd.DataFrame({
    "sample_id": expr.columns,
    "Hippo_YAP_score": hippo_score.values
}).merge(meta, on="sample_id")

display(scores.head())


# **KEY RESULT 1: Hippo/YAP score in IBD vs Control**

(Non-parametric test because sample sizes are unequal)




In [ ]:
ibd_scores = scores.loc[scores["group"].isin(["UC","CD"]), "Hippo_YAP_score"]
ctrl_scores = scores.loc[scores["group"]=="Control", "Hippo_YAP_score"]

stat, pval = mannwhitneyu(ibd_scores, ctrl_scores, alternative="two-sided")

print(f"Mann–Whitney U p-value: {pval:.3e}")

plt.figure(figsize=(6,4))
scores.boxplot(column="Hippo_YAP_score", by="group")
plt.title(f"Hippo/YAP target activity in IBD vs Control\np = {pval:.2e}")
plt.suptitle("")
plt.ylabel("Hippo/YAP target score")
plt.tight_layout()
plt.savefig(f"{FIG}/IBD_vs_Control_Hippo_score.png", dpi=200)
plt.show()


**Compute Core Signature score**

In [ ]:
# keep only core genes that exist
core_present = [g for g in core if g in expr.index]
print("Core genes used:", len(core_present))

core_score = geneset_score(expr, core_present)

scores["Core_signature_score"] = core_score.values
scores.head()


# **KEY RESULT 2: Correlation between Hippo/YAP and Core Signature**

In [ ]:
rho, p_corr = spearmanr(
    scores["Hippo_YAP_score"],
    scores["Core_signature_score"]
)

print(f"Spearman rho = {rho:.3f}")
print(f"P-value      = {p_corr:.3e}")

plt.figure(figsize=(5,5))
plt.scatter(scores["Hippo_YAP_score"], scores["Core_signature_score"], alpha=0.6)

plt.xlabel("Hippo/YAP target score")
plt.ylabel("Core signature score")
plt.title(f"Hippo activity correlates with dysbiosis–tumor core signature\n"
          f"Spearman ρ = {rho:.2f}, p = {p_corr:.2e}")

plt.tight_layout()
plt.savefig(f"{FIG}/Hippo_vs_CoreSignature_scatter.png", dpi=200)
plt.show()


**Better** **Visualizations**

In [ ]:
rho, p_corr = spearmanr(
    scores["Hippo_YAP_score"],
    scores["Core_signature_score"]
)

print(f"Spearman rho = {rho:.3f}")
print(f"P-value      = {p_corr:.3e}")

plt.figure(figsize=(5,5))
plt.scatter(scores["Hippo_YAP_score"], scores["Core_signature_score"], alpha=0.6)

plt.xlabel("Hippo/YAP target score")
plt.ylabel("Core signature score")
plt.title(f"Hippo activity correlates with dysbiosis–tumor core signature\n"
          f"Spearman ρ = {rho:.2f}, p = {p_corr:.2e}")

plt.tight_layout()
plt.savefig(f"{FIG}/Hippo_vs_CoreSignature_scatter.png", dpi=200)
plt.show()




> We now have three logically connected results:

**IBD** **vs** **Control**: Hippo/YAP activity is higher in dysbiosis

**TCGA**: Hippo/YAP-high tumors show a distinct transcriptional program

**Bridge**: Genes shared between dysbiosis and cancer scale with Hippo/YAP activity




